# ICS 604: APPLIED DATA SCIENCE

## Exponential Smoothing

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

## Forecasting Random Walks?

Recall that a random walk is defined by the recursive relationship

$$ v_t = v_{t-1} + \epsilon_t, $$

where $\epsilon_t$ is white noise. As discussed earlier, white noise has a constant mean and variance, and its values are independently and identically distributed. A common and convenient special case is when the noise is Gaussian, $\epsilon \sim \mathcal{N}(\mu, \sigma)$, which allows us to describe the behavior of the process more precisely.

In [ ]:
plt.figure(figsize=(10, 5))

v = np.zeros(101)
for t in range(1, len(v)):
    v[t] = v[t-1] + 10 * np.random.normal(0, 1)
    
plt.plot(np.arange(101), v); 

In a random walk, the change from one period to the next is entirely driven by the noise term:

$$ v_t - v_{t-1}  = \epsilon_t. $$

Since $\epsilon_t$ is white noise, it is inherently unpredictable — we do not know its realization in advance, and in practice we often do not know its exact distribution or parameters. This unpredictability is what makes forecasting random walks fundamentally challenging.

Given this limitation, the most reasonable assumption in many situations is that the immediate future will resemble the recent past, at least in the short term. Under this logic, the best forecast for the next value is simply the current value:

$$ \hat{v}_{t} = v_{t-1}. $$

This approach is known as the naïve forecast and is, in fact, the standard benchmark for random walk models. Because the forecast does not attempt to predict the noise, the resulting forecast error is exactly the realization of the white noise term itself.

In [ ]:
plt.figure(figsize=(14, 6))

x_axis = np.arange(0, 101)

plt.subplot(2,1,1)
plt.plot(x_axis[:-1], v[:100], label="$v_{t-1}$", linewidth=5)
plt.plot(x_axis[1:], v[:100], label="$v_t$", linewidth=5)
plt.legend(fontsize=14)

plt.subplot(2,1,2)
plt.plot(x_axis[19:29], v[19:29], label="$v_{t-1}$", linewidth=5)
plt.plot(x_axis[20:30], v[19:29], label="$v_{t}$", linewidth=5)
plt.legend(fontsize=14);

### Forecasting Using a Sliding Window

In many real-world time series, the noise affecting observations can be additive and arise from multiple sources. For example, in financial markets, fluctuations may be influenced by political instability, public health developments, or unexpected environmental events. Similarly, IoT sensor data can be distorted by ambient temperature changes, electromagnetic interference, or other environmental factors. These sources of noise can compound, making the observed signal more volatile than the underlying process.

In [ ]:
plt.figure(figsize=(14, 6))

v = np.zeros(500)
other_noise = 0
t_with_other_noise = []
for t in range(1, len(v)):
    v[t] = v[t-1] + 10 * np.random.normal(0, 1)
    if np.random.binomial(1, 0.02):
        other_noise = np.random.normal(100, 100) 
        v[t] += other_noise
        t_with_other_noise.append(t-1)
        
plt.plot(v);
plt.scatter(t_with_other_noise, np.array(v)[t_with_other_noise], color="red");

Importantly, some of these noise effects are short-lived. They may introduce sharp spikes or drops — outliers — that do not reflect the long-term behavior of the system. After such disturbances, the time series often returns, or regresses, to its typical level. However, these transient deviations can significantly bias predictions if they are not handled properly, especially when forecasts rely heavily on the most recent observations.

<center><img src="https://www.dropbox.com/scl/fi/4ps743q77aa6mb9cx27sf/outlier.png?rlkey=w7ldmqcaq0ex6f7br5yf1m96a&st=ofgs96j3&dl=1" alt="drawing" style="width:500px"/></center><br>

To better capture the underlying pattern, we often aim to abstract away or smooth out the noise in the data. One common approach is to use a sliding window (or moving average), where predictions are based on the average of the most recent $n$ observations rather than a single value. This helps reduce the influence of any one noisy data point. Formally, the forecast is given by

$$ v_{t+1} = \frac{1}{n}\sum_{i=0}^{n-1}v_{t-i}. $$

By averaging across a window of past values, this method produces more stable forecasts and mitigates the impact of short-term fluctuations, making it especially useful when the data contains temporary noise or outliers.

In [ ]:
random_ts = pd.read_csv("data/rand_ts.csv", header=None, names=['val'])
random_ts

In [ ]:
plt.figure(figsize=(14, 4))

pred = random_ts[-10:].mean()

plt.subplot(1,2,1)
plt.plot(random_ts)
plt.scatter(len(random_ts)+1, random_ts.val.iloc[-1], color='r', s=100)
plt.title("Prediction based on the value at time $t-1$", fontsize=12)

plt.subplot(1,2,2)
plt.plot(random_ts)
plt.scatter(len(random_ts)+1, pred, color='r', s=100)
plt.title("Prediction based on the average of the last 10 values", fontsize=12);

## Smoothing Data

Recall that this smoothing strategy is the same idea we previously used when computing a rolling average for the $\mbox{CO}_2$ data. In that setting, we smoothed the curve before fitting a model — specifically, before estimating the power law parameters using `scipy.optimize.curve_fit`. The purpose of smoothing was to reduce short-term fluctuations and highlight the underlying trend, making the model fit more reliable and less sensitive to noise.

<center><img src="https://www.dropbox.com/scl/fi/7pn98p9d02a25qbxki484/trend_2.png?rlkey=4gs28xsts7bk66lbbhoj2qcif&st=pi1qv016&dl=1" alt="drawing" style="width:600px"/>

### Smoothing Using a Running or Moving Average

More generally, smoothing can be achieved through a running (or moving) average. For each observed value $x_i$, we compute a new value $s_i$ that incorporates information from its neighboring observations. In other words, instead of treating each point in isolation, we estimate it in the context of nearby values. This approach directly applies the sliding window concept discussed earlier.

In one common formulation, we use a centered window of size $2k+1$, where $k$ is a positive integer. The smoothed value is computed as

$$
s_i = \frac{1}{2k+1} \sum_{j=-k}^{k}x_{i+j}.
$$

For example, if $k=3$, then the window size is 7, and computing $s_{11}$ involves averaging the values $x_{8}, x_{9}, x_{10}, x_{11}, x_{12}, x_{13}, x_{14}$. This symmetric window incorporates both past and future values relative to $x_i$, providing a balanced smoothing effect.

Another common approach uses a trailing window of size $k$, which only considers current and past values:

$$
s_i = \frac{1}{k} \sum_{j=0}^{k-1}x_{i-j}.
$$

For instance, with $k=4$, computing $s_11$ involves averaging $x_{8}, x_{9}, x_{10}, x_{11}$. This version is especially useful in forecasting contexts, where future values are not yet available.

Both approaches help reduce the impact of noise and outliers, producing a smoother representation of the data and making underlying patterns easier to detect and model.

In [ ]:
plt.figure(figsize=(12, 6))
plt.xlim(0, len(random_ts))
plt.plot(random_ts)
plt.plot(random_ts['val'].rolling(window=5, center=True).mean(), linewidth=4);

In [ ]:
plt.figure(figsize=(12, 6))
plt.xlim(0, len(random_ts))
plt.plot(random_ts)
plt.plot(random_ts['val'].rolling(window=5).mean(), linewidth=4);

### Effect of the Window Size

The choice of window size plays a crucial role in how a moving average (MA) represents the underlying data. A rolling window helps smooth short-term fluctuations, making overall trends easier to see. However, when the data contains large spikes or sudden changes, the resulting curve can still appear uneven or bumpy. These irregularities occur because extreme values continue to influence the average while they remain within the window.

Another noticeable effect is the sudden jump in the moving average when a large observation either enters or exits the window. As a spike is included, it can significantly raise the average, and when it drops out, the average may decrease just as abruptly. This creates visible shifts in the curve are not necessarily reflective of gradual trends in the data but are instead a direct result of how the rolling window incorporates values.

To reduce these fluctuations, a larger window size can be used. Increasing the number of observations in the window spreads the influence of any single large value across more data points, effectively dampening its impact. As a result, the moving average curve becomes smoother and more stable, providing a clearer view of long-term trends. 

In [ ]:
plt.figure(figsize=(12, 6))
plt.xlim(0, len(random_ts))
plt.plot(random_ts)
plt.plot(random_ts['val'].rolling(window=11, center=True).mean(), linewidth=4);

In [ ]:
plt.figure(figsize=(12, 6))
plt.xlim(0, len(random_ts))
plt.plot(random_ts)
plt.plot(random_ts['val'].rolling(window=21, center=True).mean(), linewidth=4);

### Handling Outliers

When dealing with outliers, simply increasing the window size to smooth out occasional spikes can create new problems. While a larger window reduces the influence of extreme values, it may also overly smooth the curve. This excessive smoothing can mask meaningful patterns in the data, unintentionally flattening significant and legitimate changes. As a result, the moving average may lag behind the true signal, delaying the detection of important shifts or trends.

On the other hand, reducing the window size to better capture genuine changes can make the model too sensitive to fluctuations. A smaller window responds quickly, but it also tracks noise more closely, weakening the smoothing effect. This can lead to abrupt, jagged movements in the curve that do not accurately reflect the underlying behavior of the data.

<center><img src="https://www.dropbox.com/scl/fi/26my4ehqi1ymlrku2nxoy/MV_window_size.png?rlkey=5nvu2a4dtvmmth5mp3zpw54n3&st=37nl9etp&dl=1" width="600"/></center>



Another way to handle outliers without sacrificing overall smoothness is to keep a relatively large window size but assign different weights to the values within that window. Instead of treating every observation equally, a weighted approach allows certain points to have more influence on the moving average than others.

For example, values near the center of the window can be given greater importance, while those closer to the boundaries contribute less. This reduces the impact of extreme values as they enter or exit the window, minimizing sudden jumps in the curve. At the same time, it preserves the benefits of a larger window by maintaining a smoother overall trend, while still being more responsive to the most relevant data points.

<center><img src="https://www.dropbox.com/scl/fi/pi52yk95uhizib9tiarz3/partial_contributions.png?rlkey=h8w2ylz33a02qknz4jp3ktqbf&st=funsfi5w&dl=1" width="600"/></center>

### Using a Gaussian for Computing the Contributions 

Instead of assigning fixed or manually chosen weights within a moving window, we can use a more principled approach by defining weights through a function such as a Gaussian. In simpler approaches, weights are often assigned based on how far a point is from the center index. However, in this formulation, the weights are determined by how similar each observation $x_l$ is to the central value $x_i$.

The smoothed value at index $i$ is computed as:

$$ 
s_i = \sum_{j=-k}^{k}w_{i+j}x_{i+j} 
$$ 

Here, the weight $w_l$ for a point $x_l$ is calculated using a Gaussian function centered at the value of $x_i$:

$$
f(x_i, \sigma_x) = \frac{1}{\sqrt{2\pi}\sigma_x}\text{exp}\left(-\frac{1}{2}\left(\frac{x_l - x_i}{\sigma_x}\right)^2\right)
$$

This means that points with values close to $x_i$ receive higher weights, regardless of their position in the window, while points with very different values (e.g., spikes or outliers) are naturally down-weighted. As a result, the method emphasizes value similarity rather than temporal proximity, making it particularly useful for reducing the influence of outliers while preserving the local structure of the data.

#### Challenges of Using a Gaussian for Computing the Contributions

Despite its advantages, using a Gaussian weighting scheme introduces several practical challenges. One key issue is that the range of values within each window can vary significantly across the time series. Because of this variability, a fixed value of $\sigma$ may not work well in all cases, and it often becomes necessary to dynamically adjust $\sigma$ for each window to ensure the computed weights remain meaningful and not vanishingly small.

Although the method is conceptually straightforward, it can be computationally expensive in practice. For every window, the algorithm must determine an appropriate $\sigma$ and compute Gaussian weights for all points within that window. This repeated calculation increases the overall cost, especially for large datasets. Additionally, like other rolling window techniques, this approach suffers from edge effects. Data points near the beginning and end of the series do not have enough neighboring values to form a full window, leading to loss of information or the need for padding strategies. 

Finally, Gaussian-weighted moving averages focus solely on local smoothing and do not inherently account for broader structures in the data, such as trends or seasonality. As a result, while they improve local smoothing, they may still fall short in capturing more complex temporal dynamics.

## Exponential Smoothing

Exponential smoothing helps address some of the limitations found in both rolling window methods and weighted moving averages by avoiding a hard cutoff of past observations. Instead of restricting attention to only a fixed subset of recent data, it incorporates all past observations into the calculation. This allows the method to retain long-term information while still emphasizing more recent behavior in the time series.

The key idea behind exponential smoothing is that the influence of past observations does not remain constant over time. Instead, the contribution of each observation decays exponentially as it becomes older. In other words, recent observations have the strongest effect on the smoothed value, while earlier observations still contribute but with progressively smaller weights. This gradual decay creates a natural balance between responsiveness to new changes and stability over time.<br><br>

<center><img src="https://www.dropbox.com/scl/fi/4k2belfehd5z968fzodk0/previous_contributions.png?rlkey=gt02tran4u7fr51jvoaskvxsb&st=x8pxzx8d&dl=1" width="600"></center><br>

Despite its simplicity, this concept forms the foundation of many highly effective forecasting techniques in time series analysis. One of the most widely used approaches based on this idea is exponential smoothing itself, which leverages exponentially decreasing weights to produce forecasts that are both stable and responsive to changes in the underlying data.

### Forms of Exponential Smoothing

Exponential smoothing is not a one-size-fits-all technique; different variants are designed to capture different patterns present in time series data. The choice of method depends on whether the data exhibits trend, seasonality, or neither. By selecting the appropriate form, we can better model the underlying structure and produce more accurate forecasts.

**Single exponential smoothing** is the simplest form and is best suited for data that shows no clear trend or seasonal pattern. It assumes that the series fluctuates around a relatively constant level, making it appropriate for stable data with only random variation. When the data exhibits a trend but no seasonality, **double exponential smoothing** becomes more appropriate. This method extends the basic approach by incorporating a component that accounts for the direction and rate of change over time, allowing it to better track upward or downward movements in the series. For data that contains both trend and seasonality, **triple exponential smoothing** is used. This method further enhances the model by adding a seasonal component, enabling it to capture repeating patterns or cycles in addition to the overall trend. As a result, it is well-suited for more complex time series that exhibit both long-term movement and periodic fluctuations.
<br>

| Type of Smoothing            | Data type                     |
|:-----------------------------|:------------------------------|
| Single Exponential Smoothing | Neither trend nor seasonality |
| Double Exponential Smoothing | Trend but no seasonality      |
| Triple Exponential Smoothing | Trend and seasonality         |


### Single Exponential Smoothing

Single exponential smoothing computes the smoothed value $s_t$ at time $t$ by combining the current observed value $v_t$ with the previously smoothed value $s_{t-1}$. This creates a recursive formulation in which each new smoothed value is a weighted blend of the most recent observation and the historical smoothed signal.

$$
s_t = \alpha\cdot v_t + (1-\alpha)s_{t-1} ~~~~~~\text{where}~~~~~~ 0 \le \alpha \le 1
$$

In this formulation, $s_t$ represents the smoothed value at time step $t$, $v_t$ is the actual observed (unsmoothed) value at time step $t$, and $s_{t-1}$ is the smoothed value from the previous time step. The parameter $\alpha$ controls the balance between responsiveness and smoothness: higher values place more emphasis on the current observation, while lower values rely more heavily on past smoothed values.

At the extremes, the behavior of the model becomes particularly intuitive. When $\alpha =0$, the equation reduces to $s_t = s_{t-1}$, meaning the smoothed value does not react to new observations and effectively retains only historical information. On the other hand, when $\alpha =1$, the smoothed value becomes $s_t = v_t$, meaning the output follows the raw data exactly with no smoothing applied.

#### Single Exponential Smoothing: Remarks

The term “exponential” in exponential smoothing becomes clear when we expand the recursive definition of the method. Starting from
$$
s_t = \alpha\cdot v_t + (1-\alpha)s_{t-1}
$$

we can repeatedly substitute for $s_{t-1}, s_{t-2}$, and so on. Doing this reveals that the current smoothed value depends on all past observations, each multiplied by a progressively smaller weight:

$$
\begin{align*}
s_t &= \alpha\cdot v_t + (1-\alpha)s_{t-1} \\
    &= \alpha\cdot v_t + (1-\alpha)(\alpha\cdot v_{t-1} + (1-\alpha)s_{t-2})\\
    &= \alpha\cdot v_t + \alpha(1-\alpha)v_{t-1} + (1-\alpha)^2s_{t-2}\\    
    &= \alpha[v_t + (1-\alpha)v_{t-1}] + (1-\alpha)^2s_{t-2}\\
    &= \alpha[v_t + (1-\alpha)v_{t-1}] + (1-\alpha)^2(\alpha\cdot v_{t-2} + (1-\alpha)s_{t-3})\\    
    &= \alpha[v_t + (1-\alpha)v_{t-1} + (1-\alpha)^2v_{t-2}] + (1-\alpha)^3s_{t-3}\\
    & ~~ \vdots \\
    &= \alpha[v_t + (1-\alpha)v_{t-1} + (1-\alpha)^2v_{t-2}+ \cdots + (1-\alpha)^{t-1}v_{1}]+(1-\alpha)^t v_{0}\\
\end{align*}
$$

In [ ]:
alpha = 0.8
t = 10
[round((1-alpha)**i, 4) for i in range(t)]

In [ ]:
alpha = 0.2
t = 10
[round((1-alpha)**i, 4) for i in range(t)]

<br>

This expansion:
$$
\begin{align*}
s_t  &=  \alpha[v_t + (1-\alpha)v_{t-1} + (1-\alpha)^2v_{t-2}+ \cdots + (1-\alpha)^{t-1}v_{1}]+(1-\alpha)^t v_{0}\\
\end{align*}
$$

shows that every past observation contributes to the smoothed value at time $t$, but the influence of older values decreases exponentially over time. The most recent observations have the largest weights, while earlier ones are gradually discounted.

The reason this is called “exponential” smoothing is that the weights form a geometric progression, where each successive weight is a constant fraction $(1-\alpha)$ of the previous one. This discrete geometric decay is the counterpart of continuous exponential decay, which is why the method is interpreted as the discrete analogue of an exponential function.

Because the smoothed value $s_t$ already incorporates information from all previous observations in a weighted manner, it can also be used directly for forecasting. A simple and widely used forecasting rule is:

$$ F_{t+1} = s_t,$$

meaning that the next time step’s prediction is simply the current smoothed value. This approach makes the forecast more robust to outliers, since extreme values only have a temporary and diminishing effect on future predictions. Overall, exponential smoothing provides a compact way to combine all past information into a single, continuously updated estimate.

#### Single Exponential Smoothing as a Function of the Forecasting Error

Single exponential smoothing can also be interpreted in terms of forecasting errors, which provides a more intuitive understanding of how the method updates its predictions over time. Starting from the standard formulation, we can rearrange the equation to highlight how the new smoothed value adjusts the previous one:

$$
\begin{align*}
s_t &= \alpha\cdot v_t + (1-\alpha)s_{t-1} \\
    &= \alpha\cdot v_t + s_{t-1}-\alpha\cdot s_{t-1} \\
    &= s_{t-1} + \alpha (v_t - s_{t-1}). \\
\end{align*}
$$

This shows that the updated estimate is equal to the previous smoothed value plus a correction term proportional to the difference between the observed value and the previous estimate.

Since the forecast at time $t+1$ is defined as $F_{t+1} = s_t$, and the forecast at time $t$ is $F_{t} = s_{t-1}$, we can rewrite the update rule entirely in terms of forecasts:
 
$$ F_{t+1} = F_{t} + \alpha (v_t - F_t). $$

Here, the term $(v_t - F_t)$ represents the forecasting error at time $t$, i.e., the difference between the observed value and the predicted value. Denoting this error as $E$, the equation becomes:

$$ F_{t+1} = F_{t} + \alpha E. $$

This formulation makes it clear that exponential smoothing updates the forecast by taking the previous prediction and adjusting it by a fraction of the most recent error. In other words, the method “learns” from its mistakes: if the forecast was too low or too high, it corrects itself proportionally based on the magnitude of the error and the smoothing parameter $\alpha$.

#### Single Exponential Smoothing with  `Pandas.Series`

In practice, exponential smoothing can be easily implemented using the Pandas library through the `Series.ewm()` function, which stands for *Exponentially Weighted Moving*. This method provides a convenient way to compute exponentially weighted statistics directly on time series data.

For example, applying exponential weighting to a series can be done as follows:

```python
random_ts['val'].ewm(alpha=0.5)
```

This returns an `ExponentialMovingWindow` object, which stores the configuration for how the exponential weights will be applied. The parameter `alpha` controls the smoothing factor, determining how quickly older observations are down-weighted. A higher value of `alpha` makes the method more responsive to recent changes, while a lower value produces a smoother result by placing more emphasis on historical data.

It is important to note that single exponential smoothing is essentially a weighted moving average with exponentially decaying weights. The `ewm` object itself does not immediately compute the smoothed values; instead, it defines the weighting scheme that will be used.

To actually compute the smoothed time series, we apply the `.mean()` function:

```Python
random_ts['val'].ewm(alpha=0.5).mean()
```

This returns the exponentially smoothed series, where each value is computed as a weighted average of all past observations, with exponentially decreasing weights controlled by the chosen `alpha`.

In [ ]:
plt.figure(figsize=(14, 8))
plt.plot(random_ts)
plt.plot(random_ts['val'].ewm(alpha=0.5).mean(), color='r', lw=5, label="Single Exp. Smoothing")
plt.plot(random_ts['val'].rolling(window=10, center=True).mean(), color='g', lw=5, label="Rolling Window")
plt.legend(fontsize=16);

#### Impact of Parameter $\alpha$

<center><img src="https://www.dropbox.com/scl/fi/zbt0103aty4774q9q7kyh/exp_smoothing.png?rlkey=hcrivnifoaz8rha09vfqxziq5&st=1as3oh6q&dl=1" alt="drawing" style="width:600px"/>

The value of $\alpha$ directly controls how quickly the influence of past observations decays in exponential smoothing. This is illustrated in the plot, where different curves show how weights are assigned to increasingly older observations relative to the most recent one (at index 1).

When $\alpha$ is large (e.g., 0.8), the weight assigned to the most recent observation is very high, and the weights for older observations drop off sharply. This means the smoothed value is driven mostly by recent data, making the method highly responsive to changes but also more sensitive to noise. For moderate values of $\alpha$ (such as 0.4 or 0.6), the decay is more gradual. Recent observations still have higher influence, but older values continue to contribute meaningfully for longer. This produces a balance between responsiveness and stability, allowing the smoothed series to track trends while still filtering out some variability. When $\alpha$ is small (e.g., 0.2), the decay is slow, meaning past observations retain influence for a much longer period. In this case, the weighting curve is much flatter, resulting in a smoother but less responsive estimate that reacts slowly to changes in the underlying signal.

Overall, the plot highlights that $\alpha$ determines the steepness of the exponential decay: higher values concentrate weight on recent observations, while lower values distribute weight more evenly across the historical data.

In [ ]:
plt.figure(figsize=(14, 8))
plt.plot(random_ts)
plt.plot(random_ts['val'].ewm(alpha=0.2).mean(), color='r', lw=4, alpha=0.5, label=r"$\alpha = 0.2$")
plt.plot(random_ts['val'].ewm(alpha=0.8).mean(), color='g', lw=4, alpha=0.5, label=r"$\alpha = 0.8$")
plt.legend(fontsize=16);

### Double Exponential Smoothing

It is generally unwise to ignore trend information when it is present in a time series, as doing so can lead to systematically biased forecasts. Double exponential smoothing addresses this issue by explicitly incorporating a **trend** component into the forecasting process. In this sense, it can be viewed as a non-parametric way of adapting to an underlying trend structure without requiring an explicit functional form.

To account for trend, we extend the idea of exponential smoothing by introducing a second recursively updated quantity that captures how the series is changing over time. Intuitively, the trend can be thought of as a weighted average of changes in the smoothed signal since $t=0$, where more recent changes are given higher importance. This is controlled by a parameter $\beta$, which determines how quickly the trend estimate reacts to new information, similar to how $\alpha$ controls the level smoothing.

The trend component $r_t$ is computed using a recursive update similar in spirit to single exponential smoothing:

$$ r_t = \beta(s_{t} - s_{t-1}) + (1-\beta)r_{t-1}. $$

Here, the difference $(s_{t} - s_{t-1})$ represents the most recent estimated change in the level, and this is blended with the previous trend estimate $r_{t-1}$. The parameter $\beta$ determines how strongly the model reacts to recent changes in the trend.<br><br>

<center><img src="https://www.dropbox.com/scl/fi/qjklm0co1jkq7ppva2kjy/trend_example.png?rlkey=h8rab7zmz9afja0ef7onth4fu&st=9ymar8m6&dl=1" width=300/></center><br>

Once the trend is estimated, it is incorporated into the level update so that forecasts can account for both the current state and the direction of movement. The level equation is modified as:

$$  s_t = \alpha \cdot v_t + (1-\alpha)(s_{t-1} + r_{t-1}). $$

This formulation adjusts the previous level not only based on past values but also by projecting forward using the estimated trend. As a result, double exponential smoothing is able to produce forecasts that better capture systematically increasing or decreasing patterns in the data, rather than assuming a flat underlying signal.

#### Double Exponential Smoothing for Forecasting

Double exponential smoothing can be directly used for forecasting by combining the most recent estimate of the level with the most recent estimate of the trend. Once both components have been updated up to time $t$, the forecast for the next time step is obtained by projecting the level forward using the trend.

Specifically, the forecast at time $t+1$ is given by:

$$ F_{t+1} = s_t + r_{t}. $$

Here, $s_t$ represents the smoothed estimate of the current level of the series, while $r_t$ captures the estimated trend (i.e., the rate of change). By adding these two components, the method assumes that the most recent trend continues into the immediate future, allowing the forecast to adjust not only to the current value of the series but also to its direction of movement.

This simple combination makes double exponential smoothing particularly effective for time series that exhibit a consistent upward or downward trend, as it enables forecasts to evolve smoothly over time rather than remaining flat.

### Triple Exponential Smoothing

Triple exponential smoothing extends the previous ideas by additionally accounting for **seasonality** in the data. This is necessary when the time series exhibits repeating patterns at regular intervals, which cannot be captured by level and trend components alone. To model this behavior, an additional quantity is introduced to represent the seasonal effect, similar in spirit to how the trend component was added in double exponential smoothing.

The key idea is to estimate seasonality by comparing observations that occur at the same position within different seasonal cycles. For example, in a time series with yearly seasonality, all March values across different years are used to update the seasonal component for March. This allows the model to learn repeating patterns in a non-parametric way, without assuming a fixed functional form for seasonality.

The seasonal component $p_t$ is updated recursively as:

$$ p_t = \gamma(v_t - s_t) + (1-\gamma) p_{t-k}, $$

where $k$ is the length of the seasonal period. This parameter defines how far back in time we look to find the corresponding seasonal position (e.g., 12 for monthly data with yearly seasonality). In practice, $k$ can be chosen based on domain knowledge or estimated using tools such as the autocorrelation function (ACF), or by selecting the value that best fits the observed data. The parameter $\gamma$ controls how quickly the seasonal component adapts to new information, similar to how $\alpha$ and $\beta$ govern the level and trend updates.

The seasonal component at time $t$ is therefore a weighted combination of the current signal deviation $(v_t - s_t)$ and the seasonal pattern observed one full cycle ago $p_{t-k}$. This ensures that the model learns repeating patterns while still adapting gradually to changes in seasonality over time.

#### Triple Exponential Smoothing: Remarks

This approach follows the same general principle used in both single and double exponential smoothing: new estimates are formed by combining current observations with past smoothed components using exponentially decaying weights. The key difference here is that seasonality is explicitly modeled by linking observations across corresponding positions in different cycles.

Once level, trend, and seasonal components have been estimated, forecasting becomes a simple combination of all three. The forecast for the next time step is given by:

$$ F_{t+1} = s_t + r_t + p_{t+1-k}. $$

This expression reflects the idea that future values are determined by the current level of the series, its direction of change, and its seasonal position. The seasonal term is aligned with the corresponding point in the cycle for the next time step. Together, these components allow triple exponential smoothing to effectively capture complex time series structures that exhibit both trend and repeating seasonal behavior.

### Testing the Exponential Smoothing

A natural question is whether exponential smoothing performs well on the $\mbox{CO}_2$ dataset that we previously analyzed manually. This dataset is particularly suitable for evaluation because it clearly exhibits both a long-term upward trend and strong seasonal patterns. As such, it provides a good test case for assessing whether the method can effectively capture both components simultaneously.

To carry out this analysis, we use the `statsmodels` package, which provides built-in implementations of exponential smoothing methods, including support for trend and seasonal components. This allows us to apply the model in a systematic way and compare its performance against our earlier observations of the data.

In [ ]:
co2_data = pd.read_csv("data/carbon_dioxide.txt", names=["co2_val"])
co2_data.co2_val = co2_data.co2_val.astype('float64')
co2_data.head()

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(co2_data, trend='add', seasonal='add',
                             seasonal_periods=12)  # Monthly data with yearly seasonality

tes = model.fit()  # The three parameters are optimized automatically

# Make predictions
forecast = tes.forecast(24)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(co2_data, lw=2, label='Original Data')
plt.plot(tes.fittedvalues, lw=2, label='Fitted Values', linestyle='--')
plt.plot(forecast, lw=2, label='Forecast', linestyle=':', color='red')
plt.legend(loc='best')
plt.title('Triple Exponential Smoothing (Holt-Winters)');

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(co2_data, trend='add', seasonal='add',
                             seasonal_periods=12)  # Monthly data with yearly seasonality

# Manually set alpha, beta, and gamma
tes = model.fit(smoothing_level=0.005, smoothing_trend=0.2, smoothing_seasonal=0.00001)

# Make predictions
forecast = tes.forecast(24)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(co2_data, lw=2, label='Original Data')
plt.plot(tes.fittedvalues, lw=2, label='Fitted Values', linestyle='--')
plt.plot(forecast, lw=2, label='Forecast', linestyle=':', color='red')
plt.legend(loc='best')
plt.title('Triple Exponential Smoothing (Holt-Winters)');

In [ ]:
model = ExponentialSmoothing(co2_data, trend=None, seasonal=None,
                             seasonal_periods=12)  # Monthly data with yearly seasonality

es = model.fit()

# Make predictions
forecast = es.forecast(24)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(co2_data, lw=2, label='Original Data')
plt.plot(es.fittedvalues, lw=2, label='Fitted Values', linestyle='--')
plt.plot(forecast, lw=2, label='Forecast', linestyle=':', color='red')
plt.legend(loc='best')
plt.title('Triple Exponential Smoothing w/o trend & seasonality');

In [ ]:
model = ExponentialSmoothing(co2_data, trend="add", seasonal=None,
                             seasonal_periods=12)  # Monthly data with yearly seasonality

des = model.fit()

# Make predictions
forecast = des.forecast(24)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(co2_data, lw=2, label='Original Data')
plt.plot(des.fittedvalues, lw=2, label='Fitted Values', linestyle='--')
plt.plot(forecast, lw=2, label='Forecast', linestyle=':', color='red')
plt.legend(loc='best')
plt.title('Triple Exponential Smoothing with trend (no seasonality)');

In [ ]:
model = ExponentialSmoothing(co2_data, trend=None, seasonal="add",
                             seasonal_periods=12)  # Monthly data with yearly seasonality

es = model.fit()

# Make predictions
forecast = es.forecast(24)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(co2_data, lw=2, label='Original Data')
plt.plot(es.fittedvalues, lw=2, label='Fitted Values', linestyle='--')
plt.plot(forecast, lw=2, label='Forecast', linestyle=':', color='red')
plt.legend(loc='best')
plt.title('Triple Exponential Smoothing with seasonality (no trend)');